# LaStBeRu QSO x DP2

Este notebook procesa los catalogos confirmed y extended, cuenta imagenes individuales DP2 por banda y genera cutouts de coadds de 20 arcsec x 20 arcsec para los targets con cobertura.

## Configuracion e imports

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord
from astropy.visualization import ImageNormalize, AsinhStretch
from lsst.daf.butler import Butler
from lsst.rsp import get_tap_service

CATALOG_FILES = {
    "confirmed": "LaStBeRu_QSO_confirmed.csv",
    "extended": "LaStBeRu_QSO_extended.csv",
}
BUTLER_REPO = "dp02"
BUTLER_COLLECTIONS = "2.2i/runs/DP0.2"
TAP_VISIT_TABLE = "dp02_dc2_catalogs.CcdVisit"
VISIT_SEARCH_RADIUS_DEG = 11.0 / 60.0
COADD_DATASET = "deepCoadd"
BANDS = "ugrizy"
CUTOUT_SIZE_ARCSEC = 20.0
OUTPUT_TABLE = "LaStBeRu_QSO_DP2_coverage.csv"
OUTPUT_DIR = Path("LaStBeRu_QSO_DP2_cutouts")
OUTPUT_DIR.mkdir(exist_ok=True)
butler = Butler(BUTLER_REPO, collections=BUTLER_COLLECTIONS)
tap_service = get_tap_service("tap")


## Cargar los dos catalogos

In [ ]:
catalogs = []
for catalog_name, filename in CATALOG_FILES.items():
    path = Path(filename)
    if not path.exists():
        raise FileNotFoundError(f"No existe {path}")
    data = pd.read_csv(path)
    required = {"DB_JNAME", "RA", "DEC"}
    missing = required - set(data.columns)
    if missing:
        raise ValueError(f"{filename} no contiene las columnas {sorted(missing)}")
    data = data.copy()
    data["catalog"] = catalog_name
    data["target_index"] = np.arange(len(data))
    data["RA"] = pd.to_numeric(data["RA"], errors="coerce")
    data["DEC"] = pd.to_numeric(data["DEC"], errors="coerce")
    data = data.dropna(subset=["RA", "DEC"]).reset_index(drop=True)
    catalogs.append(data)
targets = pd.concat(catalogs, ignore_index=True)
targets[["catalog", "target_index", "DB_JNAME", "RA", "DEC"]].head()


## Cobertura DP2 e imagenes individuales por banda

Se cuentan los datasets calexp cuyos detectores cubren la coordenada del target. El conteo es de imagenes individuales, no de coadds.

In [ ]:
from rsp_lsst_tasks import query_visit_coverage, plot_coverage_map

targets = targets.copy()
targets["target_key"] = (
    targets["catalog"].astype(str) + ":" + targets["target_index"].astype(str)
)

coverage_counts = query_visit_coverage(
    targets,
    release="DP2",
    collection=BUTLER_COLLECTIONS,
    target_id_column="target_key",
    ra_column="RA",
    dec_column="DEC",
    batch_size=100,
    progress=True,
)

coverage = targets.merge(
    coverage_counts,
    on=["target_key", "RA", "DEC"],
    how="left",
    suffixes=("", "_query"),
)
coverage.to_csv(OUTPUT_TABLE, index=False)
print(f"Targets procesados: {len(coverage)}")
print(f"Targets con cobertura DP2: {coverage['in_dp2'].sum()}")
coverage


## Tabla resumen por catalogo

In [ ]:
band_columns = [f"n_{band}" for band in BANDS]
coverage.groupby("catalog")[band_columns + ["n_images_total", "in_dp2"]].agg(
    ["count", "sum", "mean"]
)


In [ ]:
# Mapa de targets coloreado por numero total de imagenes individuales.
plot_coverage_map(
    coverage,
    ra_column="RA",
    dec_column="DEC",
    value_column="n_images_total",
    title="DP2: numero de imagenes individuales por target",
    save_path="LaStBeRu_QSO_DP2_coverage_map.png",
)


## Cutouts de coadds para targets con cobertura DP2

In [ ]:
def plot_target_coadd_cutout(row):
    coord = SkyCoord(float(row.RA), float(row.DEC), unit="deg")
    refs = list(butler.query_datasets(
        COADD_DATASET,
        where="patch.region OVERLAPS POINT(:ra, :dec)",
        bind={"ra": float(row.RA), "dec": float(row.DEC)},
    ))
    fig = plt.figure(figsize=(18, 10))
    plotted_bands = []
    for panel, band in enumerate(BANDS, start=1):
        band_ref = next(
            (ref for ref in refs if str(ref.dataId.get("band", "")) == band),
            None,
        )
        if band_ref is None:
            ax = fig.add_subplot(2, 3, panel)
            ax.text(0.5, 0.5, f"No hay coadd {band}", ha="center", va="center")
            ax.axis("off")
            continue
        coadd = butler.get(band_ref)
        wcs = coadd.fits_wcs
        x, y = wcs.world_to_pixel(coord)
        image = np.asarray(coadd.image.array, dtype=float)
        height, width = image.shape
        if not (0 <= x < width and 0 <= y < height):
            ax = fig.add_subplot(2, 3, panel)
            ax.text(0.5, 0.5, f"Target fuera del coadd {band}", ha="center", va="center")
            ax.axis("off")
            continue
        pixel_scale_arcsec = np.mean(np.abs(wcs.pixel_scale_matrix.diagonal())) * 3600.0
        half_size_pixels = max(1, int(CUTOUT_SIZE_ARCSEC / (2 * pixel_scale_arcsec)))
        x0, x1 = max(0, int(x) - half_size_pixels), min(width, int(x) + half_size_pixels)
        y0, y1 = max(0, int(y) - half_size_pixels), min(height, int(y) + half_size_pixels)
        cutout = image[y0:y1, x0:x1]
        finite = cutout[np.isfinite(cutout)]
        ax = fig.add_subplot(2, 3, panel, projection=wcs)
        if len(finite) == 0:
            ax.text(0.5, 0.5, f"Sin pixeles validos {band}", ha="center", va="center")
            ax.axis("off")
            continue
        vmin, vmax = np.percentile(finite, [5, 99.5])
        ax.imshow(
            cutout, origin="lower", cmap="gray", extent=[x0, x1, y0, y1],
            norm=ImageNormalize(vmin=vmin, vmax=vmax, stretch=AsinhStretch(0.08)),
        )
        ax.plot(x, y, "ro", ms=8, mfc="none", mew=1.5)
        ax.set_xlim(x0, x1)
        ax.set_ylim(y0, y1)
        ax.set_title(f"{band} | {pixel_scale_arcsec:.3f} arcsec/pixel")
        ax.coords[0].set_axislabel("RA")
        ax.coords[1].set_axislabel("Dec")
        plotted_bands.append(band)
    fig.suptitle(f"{row.catalog} | {row.DB_JNAME} | {row.RA:.6f}, {row.DEC:.6f}")
    plt.tight_layout()
    output_path = OUTPUT_DIR / f"{row.catalog}_{row.target_index}_{row.DB_JNAME}.png"
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return output_path, plotted_bands

cutout_rows = []
for row in coverage[coverage["in_dp2"]].itertuples(index=False):
    output_path, plotted_bands = plot_target_coadd_cutout(row)
    cutout_rows.append({
        "catalog": row.catalog,
        "target_index": row.target_index,
        "DB_JNAME": row.DB_JNAME,
        "cutout_path": str(output_path),
        "coadd_bands": ",".join(plotted_bands),
    })
cutouts = pd.DataFrame(cutout_rows)
cutouts.to_csv("LaStBeRu_QSO_DP2_cutouts.csv", index=False)
print(f"Cutouts generados: {len(cutouts)}")


## Archivos de salida

- LaStBeRu_QSO_DP2_coverage.csv: una fila por target, con cobertura DP2 y numero de visit_image por banda.
- LaStBeRu_QSO_DP2_cutouts.csv: indice de los cutouts generados.
- LaStBeRu_QSO_DP2_cutouts/: imagenes PNG de los coadds, de 20 x 20 arcsec.